# 05 Feature Engineering

## Purpose
Create a leakage-aware modeling dataset with one row per patient. Predictors use information up to 2021-12-31, while targets use future events from 2022-01-01 through 2024-12-31.

## Imports

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"
MODELS_DIR = PROJECT_ROOT / "models"
for directory in [FIGURES_DIR, TABLES_DIR, MODELS_DIR, DATA_DIR / "processed"]:
    directory.mkdir(parents=True, exist_ok=True)

import pandas as pd

from src.data_cleaning import load_raw_data, clean_all
from src.features import build_patient_features, save_feature_table

## Data loading

In [2]:
data = clean_all(load_raw_data(DATA_DIR))

## Main feature engineering

In [3]:
feature_df = build_patient_features(
    patients=data["patients"],
    diagnoses=data["diagnoses"],
    lab_results=data["lab_results"],
    medications=data["medications"],
    outcomes=data["outcomes"],
    cutoff_date="2021-12-31",
    horizon_end="2024-12-31",
)
feature_df.head()

,patient_id,age,sex,bmi,systolic_bp,diastolic_bp,heart_rate,temperature_f,smoking_status,alcohol_use,...,prior_hospitalization_count,prior_icu_admission_count,prior_mean_los,prior_readmission_rate,future_hospitalization,future_icu_admission,future_30d_readmission,future_condition_count,new_condition_count,multimorbidity_progression
0,P0000001,66.0,M,23.5,148.0,81.0,64,98.4,former,light,...,0.0,0.0,NaN,0.0,0,0,0,3,0.0,0
1,P0000002,75.0,M,24.8,158.0,86.0,45,99.5,never,moderate,...,0.0,0.0,NaN,0.0,1,0,0,5,0.0,0
2,P0000003,82.0,M,17.8,135.0,57.0,90,98.2,never,light,...,0.0,0.0,NaN,0.0,1,0,1,1,0.0,0
3,P0000004,73.0,F,28.1,118.0,83.0,102,98.9,former,moderate,...,0.0,0.0,NaN,0.0,0,0,0,3,3.0,1
4,P0000005,86.0,F,30.6,156.0,81.0,56,98.7,never,heavy,...,0.0,0.0,NaN,0.0,0,0,0,1,1.0,0


In [4]:
feature_summary = pd.DataFrame({
    "rows": [len(feature_df)],
    "columns": [feature_df.shape[1]],
    "future_hospitalization_rate": [feature_df["future_hospitalization"].mean()],
    "future_icu_admission_rate": [feature_df["future_icu_admission"].mean()],
    "future_30d_readmission_rate": [feature_df["future_30d_readmission"].mean()],
    "multimorbidity_progression_rate": [feature_df["multimorbidity_progression"].mean()],
    "mean_baseline_condition_count": [feature_df["baseline_condition_count"].mean()],
})
feature_summary.round(4)

,rows,columns,future_hospitalization_rate,future_icu_admission_rate,future_30d_readmission_rate,multimorbidity_progression_rate,mean_baseline_condition_count
0,100000,99,0.0455,0.001,0.0068,0.1811,1.8671


In [5]:
target_counts = feature_df[[
    "future_hospitalization",
    "future_icu_admission",
    "future_30d_readmission",
    "multimorbidity_progression",
]].agg(["sum", "mean"]).T

target_counts

,sum,mean
future_hospitalization,4549.0,0.04549
future_icu_admission,103.0,0.00103
future_30d_readmission,684.0,0.00684
multimorbidity_progression,18107.0,0.18107


## Key findings

- Predictors are generated only from baseline data available on or before 2021-12-31.
- The target window is 2022-01-01 to 2024-12-31.
- All-time patient diagnosis flags are excluded from predictors to reduce leakage risk.

## Saved outputs

In [6]:
feature_path = save_feature_table(feature_df, DATA_DIR / "processed" / "modeling_dataset.csv")
feature_summary.to_csv(TABLES_DIR / "05_feature_summary.csv", index=False)
target_counts.to_csv(TABLES_DIR / "05_target_counts.csv")
print("Saved feature table to", feature_path)

Saved feature table to /Users/chinmaynaringrekar/PythonProject/Disease Progression and Multimorbidity Prediction/data/processed/modeling_dataset.csv
